# 📈 Regression Metrics: Comprehensive Guide

This notebook provides a thorough exploration of regression metrics, from basic concepts to advanced applications.
We'll cover:

1. **Fundamental Concepts**: Understanding different types of regression errors
2. **Core Metrics**: MAE, MSE, RMSE, R², MAPE and their interpretations
3. **Advanced Metrics**: Adjusted R², Explained Variance, MSLE
4. **Practical Applications**: Model comparison, residual analysis
5. **Interactive Visualizations**: Comprehensive dashboards and comparisons

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand all major regression metrics and when to use them
- Be able to implement metrics calculations from scratch
- Know how to create comprehensive evaluation dashboards
- Understand the trade-offs between different metrics
- Be able to interpret residual plots and model performance

## 🔍 Understanding Regression Errors

In regression, we predict continuous values. The **error** (or **residual**) for each prediction is:

**Error = Actual Value - Predicted Value**

Different metrics aggregate these errors in different ways:

### 📊 Error Types Visualization

| Error Type | Impact | When to Use |
|------------|--------|-------------|
| **Small errors** | Less penalty | When most predictions are close |
| **Large errors** | Heavy penalty (squared) | When outliers are costly |
| **Systematic bias** | Consistent over/under-prediction | Need to detect bias |

### 🎯 Key Concepts

- **Absolute Error**: `|actual - predicted|` - treats all errors equally
- **Squared Error**: `(actual - predicted)²` - penalizes large errors more
- **Percentage Error**: `|actual - predicted|/actual` - relative to actual values
- **Bias**: Average of `actual - predicted` - systematic over/under-prediction

In [ ]:
# Import all necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, explained_variance_score
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("📦 All libraries imported successfully!")
print("🎨 Visualization style configured!")

In [ ]:
class RegressionMetrics:
    """
    A comprehensive class for calculating and visualizing regression metrics.
    
    This class provides methods to:
    - Calculate all standard regression metrics
    - Generate visualizations (residual plots, prediction plots, etc.)
    - Compare multiple models
    - Analyze metric behavior with different data characteristics
    """
    
    def __init__(self):
        self.metrics_definitions = {
            'MAE': 'Mean Absolute Error: Average absolute difference between actual and predicted',
            'MSE': 'Mean Squared Error: Average squared difference between actual and predicted',
            'RMSE': 'Root Mean Squared Error: Square root of MSE, same units as target',
            'MAPE': 'Mean Absolute Percentage Error: Average percentage deviation',
            'R²': 'Coefficient of Determination: Proportion of variance explained by model',
            'Adjusted R²': 'Adjusted R²: R² adjusted for number of predictors',
            'Explained Variance': 'Proportion of variance in target that is predictable'
        }
    
    def safe_divide(self, numerator, denominator):
        """Safe division that handles division by zero."""
        return np.where(denominator != 0, numerator / denominator, 0)
    
    def calculate_all_metrics(self, y_true, y_pred, n_features=None):
        """Calculate comprehensive regression metrics."""
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        n = len(y_true)
        
        # Basic error metrics
        mae = np.mean(np.abs(y_true - y_pred))
        mse = np.mean((y_true - y_pred) ** 2)
        rmse = np.sqrt(mse)
        
        # Percentage errors (handle division by zero)
        mape = np.mean(np.abs(self.safe_divide(y_true - y_pred, y_true))) * 100
        
        # R-squared and variants
        ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
        ss_residual = np.sum((y_true - y_pred) ** 2)
        r2 = 1 - (ss_residual / ss_total) if ss_total != 0 else 0
        
        # Adjusted R²
        if n_features is not None and n > n_features + 1:
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
        else:
            adj_r2 = r2
            
        # Explained Variance
        explained_var = 1 - (np.var(y_true - y_pred) / np.var(y_true)) if np.var(y_true) != 0 else 0
        
        metrics = {
            'MAE': mae,
            'MSE': mse, 
            'RMSE': rmse,
            'MAPE': mape,
            'R²': r2,
            'Adjusted_R²': adj_r2,
            'Explained_Variance': explained_var,
            'Max_Error': np.max(np.abs(y_true - y_pred)),
            'Mean_Error': np.mean(y_true - y_pred)  # Bias
        }
        
        return metrics
    
    def create_metrics_dashboard(self, y_true, y_pred, n_features=None, title="Regression Metrics Dashboard"):
        """Create a comprehensive dashboard with multiple visualizations."""
        metrics = self.calculate_all_metrics(y_true, y_pred, n_features)
        
        fig = plt.figure(figsize=(16, 10))
        
        # 1. Predictions vs Actual
        ax1 = plt.subplot(2, 3, 1)
        ax1.scatter(y_true, y_pred, alpha=0.6, color='blue')
        min_val = min(np.min(y_true), np.min(y_pred))
        max_val = max(np.max(y_true), np.max(y_pred))
        ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
        ax1.set_xlabel('Actual')
        ax1.set_ylabel('Predicted')
        ax1.set_title('Predictions vs Actual')
        ax1.grid(True, alpha=0.3)
        
        # 2. Residuals Plot
        ax2 = plt.subplot(2, 3, 2)
        residuals = y_true - y_pred
        ax2.scatter(y_pred, residuals, alpha=0.6)
        ax2.axhline(y=0, color='r', linestyle='--')
        ax2.set_xlabel('Predicted')
        ax2.set_ylabel('Residuals')
        ax2.set_title('Residual Plot')
        ax2.grid(True, alpha=0.3)
        
        # 3. Error Distribution
        ax3 = plt.subplot(2, 3, 3)
        ax3.hist(residuals, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
        ax3.axvline(x=0, color='r', linestyle='--')
        ax3.set_xlabel('Residuals')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Error Distribution')
        ax3.grid(True, alpha=0.3)
        
        # 4. Metrics Summary
        ax4 = plt.subplot(2, 3, (4, 5))
        ax4.axis('tight')
        ax4.axis('off')
        
        table_data = []
        for key, value in metrics.items():
            if not np.isnan(value):
                if key == 'MAPE':
                    table_data.append([key, f'{value:.2f}%'])
                else:
                    table_data.append([key, f'{value:.4f}'])
        
        table = ax4.table(cellText=table_data, colLabels=['Metric', 'Value'],
                         cellLoc='left', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.5)
        ax4.set_title('Metrics Summary', fontsize=12, fontweight='bold')
        
        # 5. Performance Interpretation
        ax5 = plt.subplot(2, 3, 6)
        ax5.axis('off')
        
        # Interpretation text
        r2_interpretation = self.interpret_r2(metrics['R²'])
        
        interpretation_text = f"""Performance Analysis
{'='*20}

R² Interpretation: {r2_interpretation}

Key Insights:
• MAE: {metrics['MAE']:.3f}
• RMSE: {metrics['RMSE']:.3f}
• R² explains {metrics['R²']*100:.1f}% of variance
• Mean bias: {metrics['Mean_Error']:.3f}
"""
        
        ax5.text(0.1, 0.9, interpretation_text, fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
        
        plt.suptitle(title, fontsize=14, fontweight='bold')
        plt.tight_layout()
        return fig
    
    def interpret_r2(self, r2_value):
        """Provide interpretation for R² values."""
        if r2_value >= 0.9:
            return "Excellent fit"
        elif r2_value >= 0.7:
            return "Good fit"
        elif r2_value >= 0.5:
            return "Moderate fit"
        elif r2_value >= 0.3:
            return "Weak fit"
        else:
            return "Very weak fit"

# Initialize the metrics calculator
calc = RegressionMetrics()
print("🧮 RegressionMetrics class initialized successfully!")
print("📚 Available methods:")
print("  • calculate_all_metrics() - Calculate all metrics")
print("  • create_metrics_dashboard() - Comprehensive dashboard")
print("  • interpret_r2() - Interpret R² values")

## 🎯 Practical Demonstration

Let's apply our RegressionMetrics class to a real dataset. We'll use the diabetes dataset to demonstrate:

1. **Data Loading & Preparation**
2. **Model Training**
3. **Comprehensive Metrics Calculation**
4. **Interactive Visualizations**

### Dataset: Diabetes Progression
- **Objective**: Predict disease progression after one year
- **Features**: 10 baseline variables (age, sex, body mass index, etc.)
- **Target**: Continuous measure of disease progression

In [ ]:
# 📊 STEP 1: DATA LOADING AND PREPARATION
print("📊 STEP 1: Data Loading and Preparation")
print("=" * 50)

# Load the diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

print(f"Dataset shape: {X.shape}")
print(f"Target range: {y.min():.1f} to {y.max():.1f}")
print(f"Target mean: {y.mean():.1f} ± {y.std():.1f}")
print(f"Features: {diabetes.feature_names}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\\nTrain set: {X_train_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")
print("✅ Data preparation completed!")

In [ ]:
# 🤖 STEP 2: MODEL TRAINING AND PREDICTIONS
print("🤖 STEP 2: Model Training and Predictions")
print("=" * 50)

# Train a linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Get predictions
y_pred = model.predict(X_test_scaled)

print(f"Model trained successfully!")
print(f"Predictions shape: {y_pred.shape}")
print(f"Prediction range: {y_pred.min():.1f} to {y_pred.max():.1f}")
print(f"Actual range: {y_test.min():.1f} to {y_test.max():.1f}")

# Show first few predictions vs actual
comparison_df = pd.DataFrame({
    'Actual': y_test[:10],
    'Predicted': y_pred[:10],
    'Error': y_test[:10] - y_pred[:10]
})
print("\\n🔍 First 10 predictions:")
print(comparison_df.round(2))
print("✅ Model training completed!")

In [ ]:
# 📈 STEP 3: COMPREHENSIVE METRICS CALCULATION
print("📈 STEP 3: Comprehensive Metrics Calculation")
print("=" * 50)

# Calculate all metrics using our custom class
metrics = calc.calculate_all_metrics(y_test, y_pred, X_train.shape[1])

print("🏆 DETAILED METRICS RESULTS:")
print("-" * 40)

print("📊 Error Metrics:")
print(f"  Mean Absolute Error (MAE):     {metrics['MAE']:.3f}")
print(f"  Mean Squared Error (MSE):      {metrics['MSE']:.3f}")
print(f"  Root Mean Squared Error (RMSE): {metrics['RMSE']:.3f}")
print(f"  Maximum Error:                 {metrics['Max_Error']:.3f}")

print("\\n🎯 Percentage Metrics:")
print(f"  Mean Absolute Percentage Error: {metrics['MAPE']:.2f}%")

print("\\n🔍 Variance Metrics:")
print(f"  R² (Coefficient of Determination): {metrics['R²']:.4f}")
print(f"  Adjusted R²:                      {metrics['Adjusted_R²']:.4f}")
print(f"  Explained Variance Score:         {metrics['Explained_Variance']:.4f}")

print("\\n⚖️ Bias Analysis:")
print(f"  Mean Error (Bias):              {metrics['Mean_Error']:.3f}")
bias_direction = "over-predicting" if metrics['Mean_Error'] < 0 else "under-predicting"
print(f"  Model tendency: {bias_direction}")

print("\\n💡 INTERPRETATION:")
print(f"  • Model explains {metrics['R²']*100:.1f}% of variance in target")
print(f"  • Typical prediction error: ±{metrics['RMSE']:.1f} units")
print(f"  • Average absolute error: {metrics['MAE']:.1f} units")
r2_interp = calc.interpret_r2(metrics['R²'])
print(f"  • Overall performance: {r2_interp}")

In [ ]:
# 📊 STEP 4: COMPREHENSIVE DASHBOARD
print("📊 STEP 4: Creating Comprehensive Metrics Dashboard")
print("=" * 50)

# Create the complete dashboard with all visualizations
dashboard_fig = calc.create_metrics_dashboard(
    y_test, y_pred, X_train.shape[1],
    title="🏥 Diabetes Progression Prediction - Complete Dashboard"
)
plt.show()

print("✅ Complete metrics dashboard created!")
print("\\n🎉 The dashboard includes:")
print("  • Predictions vs Actual scatter plot")
print("  • Residual analysis plot")
print("  • Error distribution histogram") 
print("  • Detailed metrics table")
print("  • Performance interpretation")

In [ ]:
# 🏆 MODEL COMPARISON
print("🏆 MODEL COMPARISON")
print("=" * 50)

# Define multiple models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Compare all models
results = {}

for name, model in models.items():
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Get predictions
    y_pred_model = model.predict(X_test_scaled)
    
    # Calculate metrics
    metrics_model = calc.calculate_all_metrics(y_test, y_pred_model, X_train.shape[1])
    results[name] = metrics_model

# Create comparison DataFrame
comparison_df = pd.DataFrame(results).T

# Select key metrics for comparison
key_metrics = ['MAE', 'RMSE', 'R²', 'MAPE', 'Adjusted_R²']
comparison_summary = comparison_df[key_metrics].round(4)

print("📊 MODEL COMPARISON RESULTS:")
print("=" * 60)
print(comparison_summary)

# Find best model for each metric
print("\\n🥇 BEST PERFORMING MODELS:")
print("-" * 40)
for metric in key_metrics:
    if metric in ['MAE', 'RMSE', 'MAPE']:  # Lower is better
        best_model = comparison_summary[metric].idxmin()
        best_value = comparison_summary[metric].min()
    else:  # Higher is better (R², Adjusted R²)
        best_model = comparison_summary[metric].idxmax()
        best_value = comparison_summary[metric].max()
    
    print(f"{metric:15s}: {best_model:20s} ({best_value:.4f})")

# Overall recommendation
print("\\n💡 RECOMMENDATION:")
r2_best = comparison_summary['R²'].idxmax()
print(f"Best overall model: {r2_best} (R²: {comparison_summary.loc[r2_best, 'R²']:.4f})")

## 🎓 Key Takeaways and Best Practices

### ✅ **What We've Learned:**

1. **Metric Selection**: Different metrics capture different aspects of model performance
2. **Error Interpretation**: MAE vs RMSE ratio indicates presence of outliers
3. **Bias Detection**: Mean_Error reveals systematic over/under-prediction
4. **Visualization**: Residual plots reveal patterns in model errors
5. **Model Comparison**: Multiple metrics provide comprehensive evaluation

### 📊 **Metric Selection Guide:**

| **Use Case** | **Primary Metrics** | **Why** |
|--------------|-------------------|---------|
| General Purpose | RMSE, R² | Most interpretable and widely used |
| Outlier Robust | MAE, Median AE | Less sensitive to extreme values |
| Relative Errors | MAPE, SMAPE | When error magnitude depends on target size |
| Model Selection | Adjusted R², CV Score | Accounts for model complexity |
| Business Impact | Custom weighted metrics | Align with business costs |

### 🔍 **Practical Guidelines:**

1. **Always use multiple metrics** - no single metric tells the complete story
2. **Check residual plots** - patterns indicate model problems
3. **Monitor bias** - consistent over/under-prediction needs attention
4. **Consider business context** - align metrics with real-world costs
5. **Validate on holdout data** - ensure generalization

### 🚀 **Next Steps:**
- Practice with different datasets and problem types
- Learn about cross-validation for robust evaluation
- Explore ensemble methods for better performance
- Study regularization techniques for overfitting
- Implement custom metrics for specific business needs